In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'c_models'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# runtime libraries
import copy
import hashlib
import json
import time
import math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import display
import sys
import os
import fcntl
import functools

In [ ]:
# random streams; every initializer is 63, streams are advanced rather than reseeded to 64, 65, ...
def numpy_stream(cfg, stream):
    return np.random.Generator(np.random.PCG64(cfg['seed']).jumped(stream))

def torch_stream(cfg, device, stream):
    generator_device = device.type if device.type != 'mps' else 'cpu'
    generator = torch.Generator(device=generator_device).manual_seed(cfg['seed'])
    for _ in range(stream):
        torch.rand(4096, device=generator_device, generator=generator)
    return generator

def uniform_like(value, generator):
    return torch.rand(value.shape, device=generator.device, generator=generator).to(value.device)

def initialize_stream(cfg, replicate):
    torch.manual_seed(cfg['seed'])
    for _ in range(replicate):
        torch.rand(4096)

In [ ]:
# addressed Moore ports; outgoing[..., k] points toward OFFSETS[k]
OFFSETS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
OPPOSITE = [7, 6, 5, 4, 3, 2, 1, 0]

def inbox(outgoing):
    height, width = outgoing.shape[1:3]
    padded = F.pad(outgoing, (0, 0, 1, 1, 1, 1))
    return torch.stack([padded[:, 1 + dy:1 + dy + height, 1 + dx:1 + dx + width, OPPOSITE[k]]
                        for k, (dy, dx) in enumerate(OFFSETS)], -1)

def mlp(inputs, width, outputs, zero_output=False, final_tanh=False, activation='tanh'):
    nonlinearity = nn.ReLU() if activation == 'relu' else nn.Tanh()
    layers = [nn.Linear(inputs, width), nonlinearity, nn.Linear(width, outputs)]
    if zero_output:
        nn.init.zeros_(layers[-1].weight)
        nn.init.zeros_(layers[-1].bias)
    if final_tanh:
        layers.append(nn.Tanh())
    return nn.Sequential(*layers)

def hard_bit(logit, stochastic, cfg, generator, training, flip_rate=0.0, muted=False):
    smooth = torch.sigmoid(logit / cfg['temperature'])
    probability = cfg['bernoulli_floor'] + (1 - 2 * cfg['bernoulli_floor']) * smooth if stochastic else smooth
    hard = (uniform_like(probability, generator) < probability).to(logit.dtype) if stochastic else (logit > 0).to(logit.dtype)
    # parentheses keep the forward value exactly binary; surrogate only changes the backward pass
    bit = hard + (probability - probability.detach()) if training else hard
    if flip_rate:
        flipped = uniform_like(bit, generator) < flip_rate
        bit = torch.where(flipped, 1 - bit, bit)
    if muted:
        bit = bit * 0
    return bit, probability
# only the continuous positive control reads neighbor state vectors
def neighbor_states(state):
    height, width = state.shape[1:3]
    padded = F.pad(state, (0, 0, 1, 1, 1, 1))
    return torch.stack([padded[:, 1 + dy:1 + dy + height, 1 + dx:1 + dx + width, :]
                        for dy, dx in OFFSETS], -2)

def transmit(logit, stochastic, cfg, generator, training, edge_mask, activity, flip_rate, muted):
    bit, probability = hard_bit(logit, stochastic, cfg, generator, training)
    if activity is not None:
        bit = bit * activity
        probability = probability * activity
    # errors can turn a dormant sender's zero into a false positive at the receiver
    if flip_rate:
        bit = torch.where(uniform_like(bit, generator) < flip_rate, 1 - bit, bit)
    if muted:
        bit = bit * 0
    return bit * edge_mask, probability

In [ ]:
# pointwise shared MLPs; private state and binary inbox are the only inputs for the six restricted designs
class BinaryInboxNCA(nn.Module):
    def __init__(self, spec, cfg, width):
        super().__init__()
        self.spec, self.cfg, self.width = spec, cfg, width
        self.rounds = 2 if spec['family'] == 'query_answer' else 1
        self.payload_bits = 32 * cfg['state_dim'] if spec['family'] == 'continuous' else 1
        own_dim = cfg['state_dim'] + cfg['cue_dim']
        network = functools.partial(mlp, activation=cfg.get('activation', 'tanh'))
        directions = torch.tensor(OFFSETS, dtype=torch.float32)
        self.register_buffer('directions', directions.view(1, 1, 1, 8, 2))
        self.register_buffer('edge_mask', inbox(torch.ones(1, cfg['grid'], cfg['grid'], 8)))
        if spec['family'] == 'continuous':
            self.update = network(own_dim + 8 * cfg['state_dim'], width, cfg['state_dim'], zero_output=True)
        elif spec['family'] == 'message':
            self.encoder = network(own_dim + 2, width, 1)
            self.update = network(own_dim + 8, width, cfg['state_dim'], zero_output=True)
        else:
            self.q_encoder = network(own_dim + 2, width, 1)
            self.receiver = network(own_dim + 3, width, width, final_tanh=True)
            self.r_encoder = network(width, width, 1)
            self.update = network(own_dim + 24, width, cfg['state_dim'], zero_output=True)

    def forward(self, state, cue, generator, flips=None, mute=(), inspect=False):
        flips = {} if flips is None else flips
        own = torch.cat([state, cue], -1)
        quiescent = self.cfg.get('quiescent_cells', False)
        active = ((state.abs().amax(-1) > 1e-6) | (cue[..., 0] > 0)) if quiescent else None
        trace = {}
        if self.spec['family'] == 'continuous':
            neighbors = neighbor_states(state)
            if 'continuous' in mute:
                neighbors = neighbors * 0
            update_input = torch.cat([own, neighbors.flatten(-2)], -1)
        else:
            ports = own.unsqueeze(-2).expand(-1, -1, -1, 8, -1)
            directions = self.directions.expand(*ports.shape[:-1], 2)
            addressed = torch.cat([ports, directions], -1)
            activity = active.unsqueeze(-1) if quiescent else None
            if self.spec['family'] == 'message':
                sent, probability = transmit(self.encoder(addressed).squeeze(-1), self.spec['message_stochastic'],
                                              self.cfg, generator, self.training, self.edge_mask, activity,
                                              flips.get('m', 0.0), 'm' in mute)
                received = inbox(sent)
                update_input = torch.cat([own, received], -1)
                if inspect:
                    trace = {'m': sent, 'm_in': received, 'p_m': probability}
            else:
                query, q_probability = transmit(self.q_encoder(addressed).squeeze(-1), self.spec['query_stochastic'],
                                                  self.cfg, generator, self.training, self.edge_mask, activity,
                                                  flips.get('q', 0.0), 'q' in mute)
                query_in = inbox(query)
                received = self.receiver(torch.cat([ports, directions, query_in.unsqueeze(-1)], -1))
                answering = (active.unsqueeze(-1) | query_in.bool()) if quiescent else None
                response, r_probability = transmit(self.r_encoder(received).squeeze(-1), self.spec['response_stochastic'],
                                                     self.cfg, generator, self.training, self.edge_mask, answering,
                                                     flips.get('r', 0.0), 'r' in mute)
                response_in = inbox(response)
                update_input = torch.cat([own, query, query_in, response_in], -1)
                if inspect:
                    trace = {'q': query, 'q_in': query_in, 'r': response, 'r_in': response_in,
                             'p_q': q_probability, 'p_r': r_probability}
        delta = self.update(update_input)
        if quiescent:
            # neutral input has exactly zero update; gradients still reach zero-valued received bits
            delta = delta - self.update(torch.zeros_like(update_input))
        next_state = (state + self.cfg['step_size'] * torch.tanh(delta)).clamp(-self.cfg['state_bound'], self.cfg['state_bound'])
        return (next_state, trace) if inspect else next_state

In [ ]:
# approximately match trainable parameters; same width for stochastic variants within each family
def parameter_count(cfg, family, width):
    own = cfg['state_dim'] + cfg['cue_dim']
    linear = lambda n_in, n_out: (n_in + 1) * n_out
    if family == 'continuous':
        return linear(own + 8 * cfg['state_dim'], width) + linear(width, cfg['state_dim'])
    if family == 'message':
        return linear(own + 2, width) + linear(width, 1) + linear(own + 8, width) + linear(width, cfg['state_dim'])
    return (linear(own + 2, width) + linear(width, 1) + linear(own + 3, width) + linear(width, width)
            + linear(width, width) + linear(width, 1) + linear(own + 24, width) + linear(width, cfg['state_dim']))

def make_model(spec, cfg, device, replicate):
    initialize_stream(cfg, replicate)
    width = min(range(4, 257), key=lambda w: abs(parameter_count(cfg, spec['family'], w) - cfg['parameter_target']))
    return BinaryInboxNCA(spec, cfg, width).to(device)

def parameter_macs(model):
    linear_macs = lambda module: sum(layer.in_features * layer.out_features for layer in module.modules() if isinstance(layer, nn.Linear))
    update_macs = linear_macs(model.update) * (2 if model.cfg.get('quiescent_cells', False) else 1)
    if model.spec['family'] == 'continuous':
        return update_macs
    if model.spec['family'] == 'message':
        return 8 * linear_macs(model.encoder) + update_macs
    return 8 * (linear_macs(model.q_encoder) + linear_macs(model.receiver) + linear_macs(model.r_encoder)) + update_macs

def steps_for(model, regime, reference_steps):
    return reference_steps if regime == 'updates' else reference_steps // model.rounds

def bit_cost(model, steps, batch=1):
    return int(model.edge_mask.sum().item()) * model.rounds * model.payload_bits * steps * batch

In [ ]:
# seed-only controls; p/u are a probability and latent draw, not a class label
def make_cue(pack, condition_ids, probabilities, latents, cfg, device):
    ids = torch.as_tensor(condition_ids, dtype=torch.long)
    count = len(ids)
    coordinates = pack['coordinates'][ids].to(device)
    values = torch.cat([torch.ones(count, 1), pack['cue_values'][ids],
                        torch.as_tensor(probabilities, dtype=torch.float32).reshape(count, 1),
                        torch.as_tensor(latents, dtype=torch.float32).reshape(count, 1)], -1).to(device)
    cue = torch.zeros(count, cfg['grid'], cfg['grid'], cfg['cue_dim'], device=device)
    cue[torch.arange(count, device=device), coordinates[:, 0], coordinates[:, 1]] = values
    return cue

def seed_state(pack, condition_ids, cfg, device):
    coordinates = pack['coordinates'][torch.as_tensor(condition_ids, dtype=torch.long)].to(device)
    state = torch.zeros(len(condition_ids), cfg['grid'], cfg['grid'], cfg['state_dim'], device=device)
    state[torch.arange(len(condition_ids), device=device), coordinates[:, 0], coordinates[:, 1], 4] = 1
    return state

def select_target(pack, condition_ids, probabilities, latents, device):
    labels = torch.as_tensor(np.asarray(latents) < np.asarray(probabilities), dtype=torch.long)
    return pack['targets'][torch.as_tensor(condition_ids, dtype=torch.long), labels].float().to(device), labels.to(device)

def run_steps(model, state, cue, steps, generator, flips=None, mute=(), record=()):
    frames = {}
    for step in range(1, steps + 1):
        state = model(state, cue, generator, flips=flips, mute=mute)
        if step in record:
            frames[step] = state.detach().cpu()
    return state, frames

In [ ]:
# foreground-weighted reconstruction; model outputs are premultiplied RGB + alpha
def detail_error(state, target):
    # RGB differences inside the foreground isolate eyes/mouth texture from silhouette overlap
    residual = state[..., :3] - target[..., :3]
    weight_x = torch.minimum(target[:, :, 1:, 3], target[:, :, :-1, 3])
    weight_y = torch.minimum(target[:, 1:, :, 3], target[:, :-1, :, 3])
    error_x = (residual[:, :, 1:] - residual[:, :, :-1]).square().sum(-1)
    error_y = (residual[:, 1:] - residual[:, :-1]).square().sum(-1)
    return ((error_x * weight_x).sum((1, 2)) + (error_y * weight_y).sum((1, 2))) / (
        3 * (weight_x.sum((1, 2)) + weight_y.sum((1, 2))).clamp_min(1e-8))

def reconstruction_loss(state, target, cfg):
    error = (state[..., :4] - target).square()
    weights = 1 + cfg['foreground_weight'] * target[..., 3:4]
    pixel = (error * weights).sum() / (4 * weights.sum())
    return pixel + cfg['detail_weight'] * detail_error(state, target).mean() if cfg.get('detail_weight', 0.0) else pixel

def image_metrics(state, target):
    raw = state[..., :4]
    rgba = raw.clamp(0, 1)
    error = (rgba - target).square()
    foreground = target[..., 3:4]
    predicted_mask, target_mask = rgba[..., 3] > 0.5, target[..., 3] > 0.5
    intersection = (predicted_mask & target_mask).sum((1, 2)).float()
    union = (predicted_mask | target_mask).sum((1, 2)).float()
    return {'detail_rmse': detail_error(rgba, target).sqrt(), 'mse': error.mean((1, 2, 3)), 'raw_mse': (raw - target).square().mean((1, 2, 3)),
            'foreground_rmse': ((error * foreground).sum((1, 2, 3)) / (4 * foreground.sum((1, 2, 3)))).sqrt(),
            'alpha_iou': intersection / union.clamp_min(1)}

def classify(state, prototypes, cfg):
    rgba = state[..., :4].clamp(0, 1)
    weights = 1 + cfg['foreground_weight'] * prototypes[..., 3:4].amax(1, keepdim=True)
    distances = ((rgba[:, None] - prototypes).square() * weights).mean((2, 3, 4))
    separation = ((prototypes[:, 0] - prototypes[:, 1]).square() * weights[:, 0]).mean((1, 2, 3)).clamp_min(1e-8)
    winner = distances.argmin(1)
    best = distances.min(1).values / separation
    margin = (distances.max(1).values - distances.min(1).values) / separation
    nearest = prototypes[torch.arange(len(winner), device=winner.device), winner]
    iou = image_metrics(state, nearest)['alpha_iou']
    valid = (best < cfg['quality_ratio_max']) & (margin > cfg['class_margin_min']) & (iou >= cfg['alpha_iou_min'])
    return winner, valid, best, margin

def metric_rows(state, target, metadata):
    values = {key: value.detach().cpu().numpy() for key, value in image_metrics(state, target).items()}
    frame = pd.DataFrame(values)
    for key, value in metadata.items():
        frame[key] = value
    return frame

In [ ]:
# external damage; all private state in an erased disk is removed, seed cue remains environmental input
def damage(state, rng, cfg, probability):
    count, height, width = state.shape[:3]
    yy, xx = torch.meshgrid(torch.arange(height, device=state.device), torch.arange(width, device=state.device), indexing='ij')
    keep = torch.ones(count, height, width, 1, device=state.device)
    for index in range(count):
        if rng.random() < probability:
            y, x = rng.integers(0, height), rng.integers(0, width)
            radius = max(1, int(cfg['damage_radius_fraction'] * height))
            keep[index, ..., 0] = ((yy - y).square() + (xx - x).square() > radius ** 2).float()
    return state * keep

def centered_damage(state, coordinates, cfg):
    height, width = state.shape[1:3]
    yy, xx = torch.meshgrid(torch.arange(height, device=state.device), torch.arange(width, device=state.device), indexing='ij')
    radius = max(1, int(cfg['damage_radius_fraction'] * height))
    y, x = coordinates[:, 0].to(state.device), coordinates[:, 1].to(state.device)
    keep = ((yy[None] - y[:, None, None]).square() + (xx[None] - x[:, None, None]).square() > radius ** 2)
    return state * keep[..., None]

In [ ]:
# fixed validation panel; stochastic channels remain sampled, no soft-message evaluation
@torch.no_grad()
def validation_score(model, pack, cfg, phase, regime, replicate):
    device = next(model.parameters()).device
    rng = numpy_stream(cfg, 2000 + replicate)
    ids = np.asarray(pack['splits']['validation'])
    ids = ids[np.linspace(0, len(ids) - 1, min(cfg['eval_cases'], len(ids)), dtype=int)]
    probabilities = np.zeros(len(ids)) if phase == 1 else np.resize(np.asarray(cfg['train_ps']), len(ids))
    latents = np.full(len(ids), 0.5) if phase == 1 else rng.uniform(size=len(ids))
    cue = make_cue(pack, ids, probabilities, latents, cfg, device)
    target, _ = select_target(pack, ids, probabilities, latents, device)
    generator = torch_stream(cfg, device, 30 + replicate)
    model.eval()
    scores = []
    for _ in range(cfg['val_trials']):
        state = seed_state(pack, ids, cfg, device)
        previous = 0
        for horizon in cfg.get('validation_horizons', [1]):
            steps = steps_for(model, regime, horizon * cfg['eval_steps'])
            state, _ = run_steps(model, state, cue, steps - previous, generator)
            scores.append(float(reconstruction_loss(state, target, cfg)))
            previous = steps
    model.train()
    return float(np.mean(scores))

In [ ]:
# atomic checkpoints and per-run locks; one writer, readers see complete files
def atomic_torch_save(value, path):
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def atomic_json(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    os.replace(temporary, path)

def exclusive_run(function):
    @functools.wraps(function)
    def locked(*args, **kwargs):
        destination = Path(kwargs.get('destination', args[6] if len(args) > 6 else ''))
        destination.mkdir(parents=True, exist_ok=True)
        with (destination / '.run.lock').open('a') as handle:
            try:
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError(f'run already has an active writer: {destination}') from None
            return function(*args, **kwargs)
    return locked

def run_signature(cfg, spec, phase, regime, replicate, data_hash, code_hash, portable=False):
    excluded = {'resume', 'allow_device_change'} | ({'device'} if portable else set())
    settings = {key: value for key, value in cfg.items() if key not in excluded}
    payload = {'config': settings, 'spec': spec, 'phase': phase, 'regime': regime, 'replicate': replicate,
               'dataset': data_hash, 'runtime': code_hash}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def cpu_weights(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def load_trained(path, spec, cfg, device, replicate):
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    model = make_model(spec, cfg, device, replicate)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    return model, checkpoint

In [ ]:
# training pool; reset examples, damaged states and live class switches share the same local update rule
@exclusive_run
def fit_run(spec, cfg, pack, phase, regime, replicate, destination, data_hash, code_hash, warm_start=None):
    destination.mkdir(parents=True, exist_ok=True)
    device = torch.device(cfg['device'])
    model = make_model(spec, cfg, device, replicate)
    rng = numpy_stream(cfg, replicate)
    generator = torch_stream(cfg, device, 10 + replicate)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])
    signature = run_signature(cfg, spec, phase, regime, replicate, data_hash, code_hash)
    portable_signature = run_signature(cfg, spec, phase, regime, replicate, data_hash, code_hash, portable=True)
    last_path, best_path = destination / 'last.pt', destination / 'best.pt'
    origin = None
    migration = []
    if last_path.exists() and not cfg['resume']:
        raise FileExistsError(f'checkpoint already exists: {last_path}; explicit resume is required')
    history, start_step, best_loss = [], 0, float('inf')
    train_ids = np.asarray(pack['splits']['train'])
    pool_ids = rng.choice(train_ids, cfg['pool_size'])
    pool_p = np.zeros(cfg['pool_size']) if phase == 1 else rng.choice(cfg['train_ps'], cfg['pool_size'])
    pool_u = np.full(cfg['pool_size'], 0.5) if phase == 1 else rng.uniform(size=cfg['pool_size'])
    pool = seed_state(pack, pool_ids, cfg, device)
    total_bits, total_updates, elapsed_previous = 0, 0, 0.0
    if cfg['resume'] and last_path.is_file():
        saved = torch.load(last_path, map_location='cpu', weights_only=False)
        if saved['signature'] != signature:
            permitted = cfg.get('allow_device_change', False) and saved.get('portable_signature') == portable_signature
            if not permitted:
                raise ValueError(f'checkpoint settings/data/code differ: {destination.name}; no checkpoint was overwritten')
        migration = saved.get('migration', [])
        origin = saved.get('origin')
        old_device = saved.get('device', str(device))
        if old_device != str(device):
            if not cfg.get('allow_device_change', False):
                raise ValueError('device change requires explicit --allow-device-change')
            migration = migration + [{'from': old_device, 'to': str(device), 'at_step': saved['step'],
                                      'channel_rng': 'reinitialized at seed 63; cross-device replay is not exact'}]
            previous_best = torch.load(best_path, map_location='cpu', weights_only=False)
            if previous_best['signature'] != saved['signature']:
                raise ValueError('best/last checkpoint signatures differ')
            previous_best.update(signature=signature, portable_signature=portable_signature, migration=migration)
            atomic_torch_save(previous_best, best_path)
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        rng.bit_generator.state = saved['numpy_rng']
        if old_device == str(device):
            generator.set_state(saved['channel_rng'])
        pool, pool_ids = saved['pool'].to(device), np.asarray(saved['pool_ids'])
        pool_p, pool_u = np.asarray(saved['pool_p']), np.asarray(saved['pool_u'])
        history, start_step, best_loss = saved['history'], saved['step'], saved['best_loss']
        total_bits, total_updates, elapsed_previous = saved['total_bits'], saved['total_updates'], saved['elapsed_seconds']
        if old_device != str(device):
            saved.update(signature=signature, portable_signature=portable_signature, device=str(device),
                         migration=migration, channel_rng=generator.get_state().cpu())
            atomic_torch_save(saved, last_path)
    elif warm_start is not None:
        initial = torch.load(warm_start, map_location='cpu', weights_only=False)
        model.load_state_dict(initial['model'])
        origin = {'path': str(warm_start), 'step': initial.get('step'),
                  'signature': initial.get('signature'), 'mode': 'weights only; fresh optimizer, pool and RNG'}
    model.train()
    started = time.perf_counter()
    epochs = cfg['steps1'] if phase == 1 else cfg['steps2']
    for step in range(start_step + 1, epochs + 1):
        selected = rng.choice(cfg['pool_size'], cfg['batch_size'], replace=False)
        reset_count = cfg['batch_size'] if step <= cfg.get('fresh_seed_steps', 0) else max(1, int(cfg['reset_fraction'] * cfg['batch_size']))
        if cfg.get('reset_worst', False):
            with torch.no_grad():
                prior_target, _ = select_target(pack, pool_ids[selected], pool_p[selected], pool_u[selected], device)
                difficulty = (pool[selected][..., :4] - prior_target).square().mean((1, 2, 3)).cpu().numpy()
            selected = selected[np.argsort(-difficulty)]
        reset_slots = selected[:reset_count]
        pool_ids[reset_slots] = rng.choice(train_ids, reset_count)
        pool_p[reset_slots] = 0 if phase == 1 else rng.choice(cfg['train_ps'], reset_count)
        pool_u[reset_slots] = 0.5 if phase == 1 else rng.uniform(size=reset_count)
        pool[reset_slots] = seed_state(pack, pool_ids[reset_slots], cfg, device)
        switch_count = 0
        if phase == 2:
            for slot in selected[reset_count:]:
                if rng.random() < cfg['switch_probability']:
                    old_label = pool_u[slot] < pool_p[slot]
                    # train clear online switches as well as mixed-probability growth from fresh seeds
                    pool_p[slot] = 0.0 if old_label else 1.0
                    pool_u[slot] = rng.uniform()
                    switch_count += 1
        ids, probabilities, latents = pool_ids[selected], pool_p[selected], pool_u[selected]
        cue = make_cue(pack, ids, probabilities, latents, cfg, device)
        target, _ = select_target(pack, ids, probabilities, latents, device)
        state = damage(pool[selected].detach(), rng, cfg, cfg['damage_probability'])
        reference_steps = int(2 * rng.integers(cfg['rollout_min'] // 2, cfg['rollout_max'] // 2 + 1))
        rollout = steps_for(model, regime, reference_steps)
        optimizer.zero_grad(set_to_none=True)
        state, _ = run_steps(model, state, cue, rollout, generator)
        reconstruction = reconstruction_loss(state, target, cfg)
        loss = reconstruction + cfg['state_penalty'] * state[..., 4:].square().mean()
        loss.backward()
        if cfg.get('normalize_gradients', False):
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.div_(parameter.grad.norm().clamp_min(1e-8))
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'], error_if_nonfinite=True)
        optimizer.step()
        pool[selected] = state.detach()
        total_bits += bit_cost(model, rollout, cfg['batch_size'])
        total_updates += rollout * cfg['batch_size']
        if step % cfg['val_every'] == 0 or step == epochs:
            val_loss = validation_score(model, pack, cfg, phase, regime, replicate)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed = elapsed_previous + time.perf_counter() - started
            row = {'step': step, 'train_loss': float(reconstruction.detach()), 'val_loss': val_loss,
                   'grad_norm': float(grad_norm), 'switches_this_batch': switch_count,
                   'total_bits': total_bits, 'total_cell_updates': total_updates * cfg['grid'] ** 2, 'seconds': elapsed}
            history.append(row)
            if val_loss < best_loss:
                best_loss = val_loss
                atomic_torch_save({'model': cpu_weights(model), 'step': step, 'val_loss': val_loss, 'signature': signature,
                            'phase': phase, 'model_name': spec['name'], 'regime': regime, 'replicate': replicate,
                            'portable_signature': portable_signature, 'device': str(device), 'migration': migration,
                            'origin': origin}, best_path)
            # validation consumes a separate stream; training generator/pool are unchanged
            saved = {'model': cpu_weights(model), 'optimizer': optimizer.state_dict(), 'pool': pool.cpu(),
                     'pool_ids': pool_ids.tolist(), 'pool_p': pool_p.tolist(), 'pool_u': pool_u.tolist(),
                     'numpy_rng': rng.bit_generator.state, 'channel_rng': generator.get_state().cpu(),
                     'step': step, 'best_loss': best_loss, 'history': history, 'signature': signature,
                     'total_bits': total_bits, 'total_updates': total_updates, 'elapsed_seconds': elapsed,
                     'portable_signature': portable_signature, 'device': str(device), 'migration': migration, 'origin': origin,
                     'phase': phase, 'model_name': spec['name'], 'regime': regime, 'replicate': replicate}
            atomic_torch_save(saved, last_path)
            history_path = destination / 'history.csv'
            pd.DataFrame(history).to_csv(destination / 'history.csv.tmp', index=False)
            os.replace(destination / 'history.csv.tmp', history_path)
            progress = {**row, 'phase': phase, 'model': spec['name'], 'regime': regime, 'replicate': replicate,
                        'total_steps': epochs, 'device': str(device), 'updated_at': pd.Timestamp.now(tz='UTC').isoformat()}
            atomic_json(progress, destination / 'progress.json')
            print(json.dumps(progress), file=sys.__stdout__, flush=True)
            display(pd.DataFrame([{**row, 'phase': phase, 'model': spec['name'], 'regime': regime, 'replicate': replicate}]))
    best = torch.load(best_path, map_location='cpu', weights_only=False)
    return {'phase': phase, 'model': spec['name'], 'family': spec['family'], 'regime': regime, 'replicate': replicate,
            'parameters': sum(p.numel() for p in model.parameters()), 'width': model.width,
            'best_step': best['step'], 'best_val_loss': best['val_loss'], 'signature': signature,
            'device': str(device), 'migration': migration, 'origin': origin, 'optimizer_steps': epochs, 'training_bits': total_bits, 'training_cell_updates': total_updates * cfg['grid'] ** 2,
            'seconds': elapsed_previous + time.perf_counter() - started, 'checkpoint': f'{destination.name}/best.pt'}

In [ ]:
# channel diagnostics; empirical entropy is a usage statistic, not an estimate of useful information
@torch.no_grad()
def channel_diagnostics(model, state, cue, generator, metadata):
    _, trace = model(state, cue, generator, inspect=True)
    mask = model.edge_mask.expand(len(state), -1, -1, -1).bool()
    rows = []
    for channel in ['m', 'q', 'r']:
        if channel in trace:
            values = trace[channel][mask]
            probabilities = trace[f'p_{channel}'][mask]
            rate = float(values.mean())
            entropy = -(rate * math.log2(max(rate, 1e-12)) + (1 - rate) * math.log2(max(1 - rate, 1e-12)))
            rows.append({**metadata, 'channel': channel, 'one_rate': rate, 'empirical_binary_entropy': entropy,
                         'mean_sampling_variance': float((probabilities * (1 - probabilities)).mean()) if model.spec[
                             {'m': 'message_stochastic', 'q': 'query_stochastic', 'r': 'response_stochastic'}[channel]] else 0.0,
                         'near_saturation_fraction': float(((probabilities < 0.01) | (probabilities > 0.99)).float().mean())})
    return rows

def panel_ids(pack, split, cfg):
    ids = np.asarray(pack['splits'][split])
    return ids[np.linspace(0, len(ids) - 1, min(cfg['eval_cases'], len(ids)), dtype=int)]

def all_flips(rate):
    return {'m': rate, 'q': rate, 'r': rate}

def paired_bootstrap(frame, group_columns, value_columns, cfg):
    # replicate is the uncertainty unit; never bootstrap individual pixels/frames as independent runs
    rng = numpy_stream(cfg, 6000)
    rows = []
    for key, group in frame.groupby(group_columns, dropna=False, sort=False):
        key = key if isinstance(key, tuple) else (key,)
        for metric in value_columns:
            values = group[metric].dropna().to_numpy()
            draws = rng.choice(values, (1000, len(values)), replace=True).mean(1) if len(values) > 1 else np.asarray([np.nan])
            rows.append({**dict(zip(group_columns, key)), 'metric': metric, 'mean': values.mean(), 'n_replicates': len(values),
                         'ci_low': np.nan if len(values) == 1 else np.quantile(draws, 0.025),
                         'ci_high': np.nan if len(values) == 1 else np.quantile(draws, 0.975)})
    return pd.DataFrame(rows)

In [ ]:
# evaluation batches use the same trained rule; no optimization or soft/mean messages
@torch.no_grad()
def generate_panel(model, pack, cfg, ids, probabilities, latents, regime, generator, flips=None, mute=(), blank=False):
    device = next(model.parameters()).device
    result = []
    for start in range(0, len(ids), cfg['batch_size']):
        stop = start + cfg['batch_size']
        batch_ids = ids[start:stop]
        cue = make_cue(pack, batch_ids, probabilities[start:stop], latents[start:stop], cfg, device)
        state = seed_state(pack, batch_ids, cfg, device)
        if blank:
            cue.zero_()
            state.zero_()
        state, _ = run_steps(model, state, cue, steps_for(model, regime, cfg['eval_steps']), generator, flips, mute)
        result.append(state)
    return torch.cat(result)

def transform_rgba(rgba, angle, dx, dy, saturation, gains):
    height, width = rgba.shape[1:3]
    yy, xx = torch.meshgrid(torch.arange(height, device=rgba.device), torch.arange(width, device=rgba.device), indexing='ij')
    center = height // 2
    x, y = xx.float() - center - dx, -(yy.float() - center - dy)
    theta = np.deg2rad(angle)
    sx = center + np.cos(theta) * x + np.sin(theta) * y
    sy = center + np.sin(theta) * x - np.cos(theta) * y
    grid = torch.stack([2 * (sx + 0.5) / width - 1, 2 * (sy + 0.5) / height - 1], -1)
    rgb, alpha = rgba[..., :3], rgba[..., 3:4]
    grey = (rgb * rgb.new_tensor([0.2126, 0.7152, 0.0722])).sum(-1, keepdim=True)
    color = (grey + saturation * (rgb - grey)) * rgb.new_tensor(gains)
    transformed = torch.cat([color.clamp(0, 1), alpha], -1)
    return F.grid_sample(transformed.permute(0, 3, 1, 2), grid[None].expand(len(rgba), -1, -1, -1),
                         align_corners=False, mode='bilinear', padding_mode='zeros').permute(0, 2, 3, 1)

In [ ]:
# architecture table; bit cost includes zeros and excludes nonexistent boundary edges
architecture_rows = []
for spec in CFG['models']:
    model = make_model(spec, CFG, torch.device('cpu'), 0)
    assert sum(p.numel() for p in model.parameters()) == parameter_count(CFG, spec['family'], model.width)
    for regime in CFG['budget_regimes']:
        steps = steps_for(model, regime, CFG['eval_steps'])
        architecture_rows.append({'model': spec['name'], 'family': spec['family'], 'regime': regime,
                                  'width': model.width, 'parameters': sum(p.numel() for p in model.parameters()),
                                  'parameter_target': CFG['parameter_target'], 'macs_per_cell_update': parameter_macs(model),
                                  'rounds_per_update': model.rounds, 'payload_bits_per_edge_round': model.payload_bits, 'steps_at_eval': steps,
                                  'bits_per_eval_image': bit_cost(model, steps)})
architecture_df = pd.DataFrame(architecture_rows)
architecture_df.to_csv(OUTPUT / 'architectures.csv', index=False)
display(architecture_df)
layout_rows = []
for spec in CFG['models']:
    model = make_model(spec, CFG, torch.device('cpu'), 0)
    modules = ['update'] if spec['family'] == 'continuous' else ['encoder', 'update'] if spec['family'] == 'message' else ['q_encoder', 'receiver', 'r_encoder', 'update']
    for name in modules:
        module = getattr(model, name)
        layers = [f'Linear({layer.in_features}, {layer.out_features})' if isinstance(layer, nn.Linear) else type(layer).__name__
                  for layer in module]
        layout_rows.append({'model': spec['name'], 'mlp': name, 'layers': ' -> '.join(layers),
                            'sharing': 'same weights at every site/port'})
pd.DataFrame(layout_rows).to_csv(OUTPUT / 'mlp_layouts.csv', index=False)
display(pd.DataFrame(layout_rows))

In [ ]:
# communication contract checks; random nonzero update heads expose influence/gradient paths
checks = []
routed = torch.zeros(1, 3, 3, 8)
routed[0, 1, 1, 4] = 1
arrived = inbox(routed)
assert arrived[0, 1, 2, 3] == 1 and arrived.sum() == 1
checks.append({'model': 'all', 'check': 'east-going bit arrives in the east neighbor west inbox; no wrap', 'passed': True})
pack = torch.load(INPUT / 'dataset.pt', weights_only=True)
for spec in [s for s in CFG['models'] if s['family'] != 'continuous']:
    model = make_model(spec, CFG, torch.device('cpu'), 0)
    with torch.no_grad():
        model.update[2].weight.normal_(0, 0.1)
        model.update[2].bias.normal_(0, 0.1)
    generator = torch_stream(CFG, torch.device('cpu'), 2)
    ids = [pack['splits']['train'][0]]
    cue = make_cue(pack, ids, [0.5], [0.2], CFG, torch.device('cpu'))
    state = seed_state(pack, ids, CFG, torch.device('cpu'))
    assert (cue.abs().sum(-1) > 0).sum() == 1
    output, trace = model(state, cue, generator, inspect=True)
    for key, value in trace.items():
        if not key.startswith('p_'):
            assert torch.equal(value, value.round()) and value.min() >= 0 and value.max() <= 1
    output.sum().backward()
    encoders = [model.encoder] if spec['family'] == 'message' else [model.q_encoder, model.receiver, model.r_encoder]
    for encoder in encoders:
        assert sum(float(p.grad.abs().sum()) for p in encoder.parameters() if p.grad is not None) > 0
    checks.append({'model': spec['name'], 'check': 'hard train-time wires, seed-only cue, encoder gradients', 'passed': True})
    model.eval()
    initial_rng = generator.get_state()
    first = model(state, cue, generator)
    generator.set_state(initial_rng)
    assert torch.equal(first, model(state, cue, generator))
    checks.append({'model': spec['name'], 'check': 'exact replay from saved channel RNG state', 'passed': True})
    stochastic = spec['message_stochastic'] or spec['query_stochastic'] or spec['response_stochastic']
    if not stochastic:
        assert torch.equal(first, model(state, cue, generator))
        checks.append({'model': spec['name'], 'check': 'deterministic forward independent of channel RNG', 'passed': True})
    else:
        assert not torch.equal(first, model(state, cue, generator))
        checks.append({'model': spec['name'], 'check': 'sampled messages change fixed-input output', 'passed': True})
    # common random numbers isolate the spatial effect of perturbing one cell
    changed = state.clone()
    cy = cx = CFG['grid'] // 2
    changed[:, cy, cx, :] += 0.5
    generator.set_state(initial_rng)
    base = model(state, cue, generator)
    generator.set_state(initial_rng)
    perturbed = model(changed, cue, generator)
    distant = torch.ones(CFG['grid'], CFG['grid'], dtype=torch.bool)
    distant[cy - 1:cy + 2, cx - 1:cx + 2] = False
    assert torch.equal(base[:, distant], perturbed[:, distant])
    checks.append({'model': spec['name'], 'check': 'one macro-step influence stays within one Moore hop', 'passed': True})
    if spec['family'] == 'query_answer':
        local = torch.randn(1, CFG['state_dim'] + CFG['cue_dim'] + 2)
        a = model.r_encoder(model.receiver(torch.cat([local, torch.zeros(1, 1)], -1)))
        b = model.r_encoder(model.receiver(torch.cat([local, torch.ones(1, 1)], -1)))
        assert not torch.equal(a, b)
        checks.append({'model': spec['name'], 'check': 'response logit depends on received query', 'passed': True})
# quiet-grid contract uses only own state/cue and received payload; no neighbor alive-mask side channel
if CFG.get('quiescent_cells', False):
    for spec in CFG['models']:
        model = make_model(spec, CFG, torch.device('cpu'), 0)
        with torch.no_grad():
            model.update[2].weight.normal_(0, 0.1)
            model.update[2].bias.fill_(0.1)
        state = seed_state(pack, ids, CFG, torch.device('cpu'))
        blank = torch.zeros_like(state)
        assert torch.equal(model(blank, torch.zeros_like(cue), generator), blank)
        mute = ('continuous',) if spec['family'] == 'continuous' else ('m',) if spec['family'] == 'message' else ('q', 'r')
        isolated = model(state, cue, generator, mute=mute)
        outside_seed = (state.abs().sum(-1) == 0) & (cue.abs().sum(-1) == 0)
        assert not isolated[outside_seed].count_nonzero()
        checks.append({'model': spec['name'], 'check': 'zero grid stays zero; muted wires cannot activate other cells', 'passed': True})
        if spec['family'] == 'continuous':
            assert model.payload_bits == 32 * CFG['state_dim']
            generator_state = generator.get_state()
            a = model(state, cue, generator)
            generator.set_state(generator_state)
            b = model(state, cue, generator, mute=mute)
            assert not torch.equal(a, b)
            checks.append({'model': spec['name'], 'check': 'positive control reads continuous neighbors; float32 payloads charged explicitly', 'passed': True})
        else:
            # an active sender's zero bit must still get a learning gradient at a dormant receiver
            send_head = model.encoder[-1] if spec['family'] == 'message' else model.r_encoder[-1]
            with torch.no_grad():
                send_head.weight.zero_()
                send_head.bias.fill_(-0.5)
            model.zero_grad(set_to_none=True)
            output = model(state, cue, generator)
            output.sum().backward()
            assert send_head.bias.grad.abs().sum() > 0
            checks.append({'model': spec['name'], 'check': 'zero-valued hard payload retains sender gradient at dormant receiver', 'passed': True})
            corrupt = {'m': 1.0} if spec['family'] == 'message' else {'q': 1.0, 'r': 1.0}
            noisy, trace = model(blank, torch.zeros_like(cue), generator, flips=corrupt, inspect=True)
            assert noisy.count_nonzero() > 0
            for key, value in trace.items():
                if not key.startswith('p_'):
                    assert torch.equal(value, value.round()) and value.min() >= 0 and value.max() <= 1
            checks.append({'model': spec['name'], 'check': 'external flips can activate dormant receivers; corrupted wires remain binary', 'passed': True})
# detail metric must reject a perfectly overlapping silhouette with a featureless interior
if CFG.get('task') == 'canonical_reconstruction':
    target = pack['targets'][0:1, 0].float()
    flat = target.clone()
    alpha = target[..., 3:4]
    flat[..., :3] = target[..., :3].sum((1, 2), keepdim=True) / alpha.sum((1, 2), keepdim=True) * alpha
    assert image_metrics(target, target)['detail_rmse'].item() == 0
    assert image_metrics(flat, target)['alpha_iou'].item() == 1
    assert detail_error(flat, target).item() > 0
    checks.append({'model': 'all', 'check': 'perfect target has zero detail error; flat oracle-mask baseline has nonzero detail error', 'passed': True})
checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUTPUT / 'contract_checks.csv', index=False)
display(checks_df)

In [ ]:
# export definitions as a notebook; downstream tasks consume this exact handoff
source_nb = nbformat.read(TASK / 'src' / 'c_models.ipynb', as_version=4)
exported = nbformat.v4.new_notebook(cells=[c for c in source_nb.cells if 'runtime' in c.metadata.get('tags', [])])
exported.metadata.kernelspec = {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}
nbformat.write(exported, OUTPUT / 'runtime.ipynb')
digest = hashlib.sha256((OUTPUT / 'runtime.ipynb').read_bytes()).hexdigest()
(OUTPUT / 'runtime.sha256').write_text(digest)